# 05 - Global analysis
Finalized on june 2nd 2026

In [ ]:
import anndata as ad
import scanpy as sc
import scanpy.external as sce
import scirpy as ir
import pandas as pd
import numpy as np
import seaborn as sb
import os
import muon as mu
from muon import prot as pt
from scipy.stats import binomtest
from scipy.stats import mannwhitneyu
from scipy.stats import tukey_hsd
from statsmodels.stats.multitest import multipletests
import io
import scrublet as scr

# Helps to define plotting area
import matplotlib.pyplot as plt
from matplotlib import rcParams 
from matplotlib.pyplot import rc_context
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib import colors
from matplotlib_venn import venn2
import matplotlib.image as mpimg

#Conversions
import biomart

# set seed
np.random.seed(44)

#display options
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999

#Global figure size: 
plt.rcParams["figure.figsize"] = (3.5, 3)

## For making text editable in .pdfs once exported use cairo backend instead of agg
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams["font.family"] = "Arial"
sb.set_theme(style="ticks", font="Arial")

# Helper functions from Mimi

In [ ]:
tcr_gene_prefixs = ['TRAV', 'TRAJ', 'TRAC', 'TRB', 'TRDV', 'TRDC', 'TRG']

def filter_tcr_genes(adata):
    non_tcr_genes = adata.var_names
    for prefix in tcr_gene_prefixs:
        non_tcr_genes = [el for el in non_tcr_genes if not el.startswith(prefix)]
    adata = adata[:, non_tcr_genes]
    return adata

def filter_high_var(adata, n):
    #sc.pp.highly_variable_genes(
    #adata,
    #flavor="seurat_v3",
    #n_top_genes=n,
    #layer="counts",
    #batch_key="pool",
    #subset=True,
    #span = 0.75) #default = 0.3. It's the fraction of cells used for estimation of hvg - if I have 150 cells in one sample but 150.000 cells in total then this sample becomes an issue. 
    
    sc.pp.highly_variable_genes(adata, n_top_genes=n)
    adata = adata[:, adata.var['highly_variable']].copy()
    return adata


def calculate_umap(adata, n_high_var=5000, remove_tcr_genes=True):
    """
    Calculates UMAP coordinates
    :param adata: AnnData object
    :param n_high_var: how many variable genes to use, None uses raw data
    :param remove_tcr_genes: whether to remove TCR genes
    :return: results stored in adata object
    """
    adata_tmp = adata.copy()

    if remove_tcr_genes:
        adata_tmp = filter_tcr_genes(adata_tmp)
    if n_high_var is not None:
        adata_tmp = filter_high_var(adata_tmp, n_high_var)

    sc.pp.neighbors(adata_tmp, n_pcs=20)            
    sc.tl.umap(adata_tmp)
    adata.obsm['X_umap'] = adata_tmp.obsm['X_umap']


def calculate_leiden(adata, resolution, n_high_var=5000, remove_tcr_genes=True):
    adata_tmp = adata.copy()

    if remove_tcr_genes:
        adata_tmp = filter_tcr_genes(adata_tmp)
    if n_high_var is not None:
        adata_tmp = filter_high_var(adata_tmp, n_high_var)

    sc.pp.neighbors(adata_tmp, n_pcs=20)
    sc.tl.leiden(adata_tmp, resolution=resolution)

    leiden_column_name = f"leiden_{resolution}"
    adata.obs[leiden_column_name] = adata_tmp.obs['leiden']

# Loading data

In [ ]:
sdata = mu.read(filename="../../04_data_objects/06_preprocessed_v2/2026_6_main_dataset_clustered_annotated.h5mu")

## How many cells?

In [ ]:
sdata["gex"].obs[["final_ann", "Compartment"]].value_counts()

In [ ]:
sdata["gex"].obs[~sdata["gex"].obs["final_ann"].isin(["CD4 T cells", "CD8 T cells", "CD4/CD8 T cells"])][["Compartment"]].value_counts().sum()

# Fix catpat (assign IC)

In [ ]:
#Group annotation
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]
IC_list = ["Pt5", "Pt4"]

#Match the lists defined above to define "RRMS" and "Control"
sdata["gex"].obs["catpat"] = pd.Categorical(np.where(sdata["gex"].obs["donor"].isin(patient_list), "MS", np.where(sdata["gex"].obs["donor"].isin(control_list), "NIC", np.where(sdata["gex"].obs["donor"].isin(IC_list), "IC", "Error"))))


# Visualize 

In [ ]:
# Set the order and palette whereby the different clusters are displayed in umaps and elsewhere

celltype_palette = {
    "CD4 T cells":      "#ffbb78",  # light orange / peach 
    "CD8 T cells":      "#1f77b4",  # blue 
    "CD4/CD8 T cells":  "#d62728",   # red  
    "MAIT cells":     "#c49c94",  # light brown / tan
    "Multiplets":     "#98df8a",   # light green
    "MG/BAMs":        "#2ca02c",  # green
    "Monocytes":      "#ff9896",   # light red / salmon             
    "DCs":            "#9467bd",   # purple
    "CD20+ B cells":  "#aec7e8",   # light blue
    "Plasmablasts":   "#8c564b",   # brown
    "NK cells":       "#c5b0d5",   # light purple / lavender
    "pDCs":           "#e377c2",   # pink      
}

# celltype_palette = {
#     "CD4 T cells":      "#ffbb78",  # light orange / peach 
#     "CD8 T cells":      "#1f77b4",  # blue 
#     "CD4/CD8 T cells":  "#d62728",   # red  
#     "MAIT cells (PB)":  "#c49c94",  # light brown / tan
#     "MAIT cells (CSF)": "#000000",   # black
#     "Multiplets":     "#98df8a",   # light green
#     "MG/BAMs":        "#2ca02c",  # green
#     "Monocytes":      "#ff9896",   # light red / salmon             
#     "DCs":            "#9467bd",   # purple
#     "CD20+ B cells":  "#aec7e8",   # light blue
#     "Plasmablasts":   "#8c564b",   # brown
#     "NK cells":       "#c5b0d5",   # light purple / lavender
#     "pDCs":           "#e377c2",   # pink      
# }

order = [
    "CD4 T cells",
    "CD8 T cells",
    "CD4/CD8 T cells",
    "MAIT cells",
    "Multiplets",
    "MG/BAMs",
    "Monocytes",
    "DCs",
    "CD20+ B cells",
    "Plasmablasts",
    "NK cells",
    "pDCs"
]

sdata["gex"].obs["final_ann"] = pd.Categorical(
    sdata["gex"].obs["final_ann"],
    categories=order,
    ordered=True
)


In [ ]:
plt.rcParams["figure.figsize"] = (5, 4)

sc.pl.umap(
    sdata["gex"],
    color=["final_ann"],
    legend_fontweight="bold",
    frameon=False,
    palette=celltype_palette,
    vmin=0,
    legend_fontoutline=2,
    size=5,
    show=False
)

# Rasterize only the UMAP points
ax = plt.gca()
for coll in ax.collections:
    coll.set_rasterized(True)

plt.savefig(
    "../../05_plots/13_final_plots/36_GlobalAnalysis_umap.pdf",
    dpi=400,
    bbox_inches="tight",
)

plt.show()

plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

In [ ]:
plt.rcParams["figure.figsize"] = (5, 4)

# Define colors for Compartment
comp_palette = {"CSF": "#4B8AC1", "PB": "#C06A6A"}

sc.pl.umap(
    sdata["gex"],
    color=["Compartment"],
    legend_fontweight="bold",
    frameon=False,
    palette=comp_palette,
    vmin=0,
    legend_fontoutline=2,
    size=5,
    show=False
)

# Rasterize only the UMAP points
ax = plt.gca()
for coll in ax.collections:
    coll.set_rasterized(True)

plt.savefig(
    "../../05_plots/13_final_plots/36_GlobalAnalysis_umap_Compartments.pdf",
    dpi=400,
    bbox_inches="tight",
)

plt.show()

plt.rcParams["figure.figsize"] = plt.rcParamsDefault["figure.figsize"]

In [ ]:
marker_genes = {
        "T cells": ["CD3E", "LCK", "TRAC", "CD4", "CD8A", "CD8B"],
        "MAIT" : ["TRAV1-2", "SLC4A10"],
        "MG/BAM":  ["TREM2", "APOE", "LYVE1", "SPP1"],
        "Monocytes" : ["FCN1", "VCAN"], 
        "mDCs": ["CD1C", "FLT3", "CLEC9A"],
        "B cells": ["MS4A1", "CD19", "IGHG1"],
        "NK cells": ["NCAM1", "NCR1"], 
        "pDCs": ["IL3RA", "PTCRA", "IRF7"],
        "APC-status": ["CST3", "CD74", "HLA-DRB1"]}

# ## Visualize
# dp = sc.pl.dotplot(sdata["gex"], marker_genes, groupby="L1_ann", standard_scale="var", return_fig=True, show=False)
# dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5)

dp = sc.pl.dotplot(
    sdata["gex"],
    marker_genes,
    groupby="final_ann",
    standard_scale="var",
    cmap="RdBu_r",
    return_fig=True,
    show=False,
)

dp.add_totals()
dp.style(dot_edge_color="black", dot_edge_lw=0.5)

# Build the figure
dp.make_figure()

# Save
dp.fig.savefig(
    "../../05_plots/13_final_plots/37_global_analysis_dotplot.pdf",
    bbox_inches="tight",
    dpi=400,
)

# Differential cell abundance

In [ ]:
# --- Prepare obs_df ---
obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[obs_df["Compartment"] == "PB"]
obs_df = obs_df[obs_df["final_ann"].isin(["CD4 T cells", "CD8 T cells", "CD4/CD8 T cells", "MAIT cells"])] #a few cells (1-3 cells per donor) are clustering with non-T cells
obs_df["donor"] = obs_df["donor"].cat.remove_unused_categories() 
obs_df["final_ann"] = obs_df["final_ann"].cat.remove_unused_categories() 

#Greyscale palette
palette = sb.color_palette("Greys", 3)
catpat_palette = dict(zip(["NIC", "IC", "MS"], palette))

# Leiden cluster
cls = "final_ann"

# Calculate counts per donor and cluster
obs_df = (
    obs_df[["donor", cls, "catpat"]]
    .groupby(["donor", cls])
    .size()
    .reset_index(name="count")
)

# Total per donor
obs_df["total"] = obs_df.groupby("donor")["count"].transform("sum")

# Percentage per donor
obs_df["cluster_pct"] = 100 * obs_df["count"] / obs_df["total"]

# Annotate catpat
patient_list = ["Pt4", "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt5", "Pt8", "Pt18", "Pt19", "Pt21"]
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "Error")
    )
)

#Group annotation
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]
IC_list = ["Pt5", "Pt4"]

#Match the lists defined above to define "RRMS" and "Control"
obs_df["catpat"] = pd.Categorical(np.where(obs_df["donor"].isin(patient_list), "MS", np.where(obs_df["donor"].isin(control_list), "NIC", np.where(obs_df["donor"].isin(IC_list), "IC", "Error"))))

## Make certain that catpat has the right order
obs_df["catpat"] = pd.Categorical(
    obs_df["catpat"],
    categories=["NIC", "IC", "MS"],
    ordered=True
)

## Transform for log-scale
obs_df["cluster_pct"] = obs_df["cluster_pct"] + 1e-2

# --- Statistics: Mann–Whitney U per cluster ---
pvals = {}
clusters = obs_df[cls].cat.categories if hasattr(obs_df[cls], "cat") else obs_df[cls].unique()

for cl in clusters:
    sub = obs_df[obs_df[cls] == cl]
    rrms = sub.loc[sub["catpat"] == "MS", "cluster_pct"]
    ctrl = sub.loc[sub["catpat"].isin(["NIC", "IC"]), "cluster_pct"]

    if (len(rrms) > 1) and (len(ctrl) > 1):
        _, p = mannwhitneyu(rrms, ctrl, alternative="two-sided", use_continuity=False)
        pvals[cl] = p
    else:
        pvals[cl] = np.nan


# --- Plot ---
plt.figure(figsize=(4, 5))

ax = sb.boxplot(
    data=obs_df,
    x=cls,
    y="cluster_pct",
    hue="catpat",
    dodge=True,
    fill=True,
    showfliers=False,
    linewidth=0.5,
    palette=catpat_palette
)

sb.stripplot(
    data=obs_df,
    x=cls,
    y="cluster_pct",
    hue="catpat",
    dodge=True,
    color="black", 
    jitter=0.2,
    alpha=1,
    size=5,
    linewidth=0.5,
    palette=catpat_palette,
    ax=ax
)

# yscale
plt.yscale("log")

# Fix double legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:3], labels[:3], title="catpat", frameon=False)

ax.set_xlabel("Leiden cluster")
ax.set_ylabel("Percentage")
plt.xticks(rotation=45, ha="right")

# --- Correction or multiple hypothesis testing ---
pvals_df = pd.DataFrame({
    cls: list(pvals.keys()),
    "p_value": list(pvals.values())
})

pvals_df["p_adj"] = multipletests(
    pvals_df["p_value"].fillna(1),
    method="fdr_bh"
)[1]

def p_to_star(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

p_lookup = dict(zip(pvals_df[cls], pvals_df["p_adj"]))

# --- Annotate p-values ---
ymax = obs_df.groupby(cls)["cluster_pct"].max()

for i, cl in enumerate(clusters):
    p = p_lookup.get(cl, np.nan)

    if np.isnan(p):
        continue

    y = ymax[cl] * 1.08

    ax.text(
        i, y,
        p_to_star(p),
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )

    ax.plot(
        [i - 0.2, i + 0.2],
        [y * 0.98, y * 0.98],
        lw=1,
        c="black"
    )

sb.despine()
plt.grid(axis="y")
plt.tight_layout()

# Save
plt.savefig(
    "../../05_plots/13_final_plots/47_CellAbundances_PB.pdf",
    bbox_inches="tight",
    dpi=400,
)
plt.show()

print(pvals_df)


In [ ]:
# --- Prepare obs_df ---
obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[obs_df["Compartment"] == "CSF"]
obs_df["donor"] = obs_df["donor"].cat.remove_unused_categories() 

#Greyscale palette
palette = sb.color_palette("Greys", 3)
catpat_palette = dict(zip(["NIC", "IC", "MS"], palette))

# Leiden cluster
cls = "final_ann"

# Calculate counts per donor and cluster
obs_df = (
    obs_df[["donor", cls, "catpat"]]
    .groupby(["donor", cls])
    .size()
    .reset_index(name="count")
)

# Total per donor
obs_df["total"] = obs_df.groupby("donor")["count"].transform("sum")

# Percentage per donor
obs_df["cluster_pct"] = 100 * obs_df["count"] / obs_df["total"]

# Annotate catpat
patient_list = ["Pt4", "Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt5", "Pt8", "Pt18", "Pt19", "Pt21"]
obs_df["catpat"] = pd.Categorical(
    np.where(
        obs_df["donor"].isin(patient_list), "RRMS",
        np.where(obs_df["donor"].isin(control_list), "Control", "Error")
    )
)

#Group annotation
patient_list = ["Pt11", "Pt12", "Pt14", "Pt17", "Pt20"]
control_list = ["Pt1", "Pt8", "Pt18", "Pt19", "Pt21"]
IC_list = ["Pt5", "Pt4"]

#Match the lists defined above to define "RRMS" and "Control"
obs_df["catpat"] = pd.Categorical(np.where(obs_df["donor"].isin(patient_list), "MS", np.where(obs_df["donor"].isin(control_list), "NIC", np.where(obs_df["donor"].isin(IC_list), "IC", "Error"))))

## Make certain that catpat has the right order
obs_df["catpat"] = pd.Categorical(
    obs_df["catpat"],
    categories=["NIC", "IC", "MS"],
    ordered=True
)

## Transform for log-scale
obs_df["cluster_pct"] = obs_df["cluster_pct"] + 1e-2

# --- Statistics: Mann–Whitney U per cluster ---
pvals = {}
clusters = obs_df[cls].cat.categories if hasattr(obs_df[cls], "cat") else obs_df[cls].unique()

for cl in clusters:
    sub = obs_df[obs_df[cls] == cl]
    rrms = sub.loc[sub["catpat"] == "MS", "cluster_pct"]
    ctrl = sub.loc[sub["catpat"].isin(["NIC", "IC"]), "cluster_pct"]

    if (len(rrms) > 1) and (len(ctrl) > 1):
        _, p = mannwhitneyu(rrms, ctrl, alternative="two-sided", use_continuity=False)
        pvals[cl] = p
    else:
        pvals[cl] = np.nan


# --- Plot ---
plt.figure(figsize=(9, 5))

ax = sb.boxplot(
    data=obs_df,
    x=cls,
    y="cluster_pct",
    hue="catpat",
    dodge=True,
    fill=True,
    showfliers=False,
    linewidth=0.5,
    palette=catpat_palette
)

sb.stripplot(
    data=obs_df,
    x=cls,
    y="cluster_pct",
    hue="catpat",
    dodge=True,
    color="black", 
    jitter=0.2,
    alpha=1,
    size=5,
    linewidth=0.5,
    palette=catpat_palette,
    ax=ax
)

# yscale
plt.yscale("log")

# Fix double legend
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:3], labels[:3], title="catpat", frameon=False)

ax.set_xlabel("Leiden cluster")
ax.set_ylabel("Percentage")
plt.xticks(rotation=45, ha="right")

# --- Correction or multiple hypothesis testing ---
pvals_df = pd.DataFrame({
    cls: list(pvals.keys()),
    "p_value": list(pvals.values())
})

pvals_df["p_adj"] = multipletests(
    pvals_df["p_value"].fillna(1),
    method="fdr_bh"
)[1]

def p_to_star(p):
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

p_lookup = dict(zip(pvals_df[cls], pvals_df["p_adj"]))

# --- Annotate p-values ---
ymax = obs_df.groupby(cls)["cluster_pct"].max()

for i, cl in enumerate(clusters):
    p = p_lookup.get(cl, np.nan)

    if np.isnan(p):
        continue

    y = ymax[cl] * 1.08

    ax.text(
        i, y,
        p_to_star(p),
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )

    ax.plot(
        [i - 0.2, i + 0.2],
        [y * 0.98, y * 0.98],
        lw=1,
        c="black"
    )

sb.despine()
plt.grid(axis="y")
plt.tight_layout()

# Save
plt.savefig(
    "../../05_plots/13_final_plots/47_CellAbundances_CSF.pdf",
    bbox_inches="tight",
    dpi=400,
)
plt.show()

print(pvals_df)


### Use Tukey's test for multiple comparisons
ChatGPT helped me write the code for the grouped mannwhitney U test above. It did not handle all the assumptions correctly (fx. here we are working with continueous data). Furthermore, we need a test that can do multiple comparisons and not just NIC/IC vs MS. scipy has access to tukey's test, which is parametric. Unfortunately, no non-parametric tests specifically for multiple comparisons was present in scipy. 
https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.tukey_hsd.html#scipy.stats.tukey_hsd

In [ ]:
## Define the group you want to test first: 
testA = obs_df[obs_df["final_ann"] == "CD4 T cells"]
group0 = testA[testA["catpat"] == "NIC"]["cluster_pct"]
group1 = testA[testA["catpat"] == "IC"]["cluster_pct"]
group2 = testA[testA["catpat"] == "IC"]["cluster_pct"]
res = tukey_hsd(group0, group1, group2)
print(res)

In [ ]:
## Define the group you want to test first: 
testA = obs_df[obs_df["final_ann"] == "CD8 T cells"]
group0 = testA[testA["catpat"] == "NIC"]["cluster_pct"]
group1 = testA[testA["catpat"] == "IC"]["cluster_pct"]
group2 = testA[testA["catpat"] == "IC"]["cluster_pct"]
res = tukey_hsd(group0, group1, group2)
print(res)

In [ ]:
## Define the group you want to test first: 
testA = obs_df[obs_df["final_ann"] == "CD4/CD8 T cells"]
group0 = testA[testA["catpat"] == "NIC"]["cluster_pct"]
group1 = testA[testA["catpat"] == "IC"]["cluster_pct"]
group2 = testA[testA["catpat"] == "IC"]["cluster_pct"]
res = tukey_hsd(group0, group1, group2)
print(res)

In [ ]:
## Define the group you want to test first: 
testA = obs_df[obs_df["final_ann"] == "MAIT cells"]
group0 = testA[testA["catpat"] == "NIC"]["cluster_pct"]
group1 = testA[testA["catpat"] == "IC"]["cluster_pct"]
group2 = testA[testA["catpat"] == "IC"]["cluster_pct"]
res = tukey_hsd(group0, group1, group2)
print(res)

In [ ]:
## Define the group you want to test first: 
testA = obs_df[obs_df["final_ann"] == "Multiplets"]
group0 = testA[testA["catpat"] == "NIC"]["cluster_pct"]
group1 = testA[testA["catpat"] == "IC"]["cluster_pct"]
group2 = testA[testA["catpat"] == "IC"]["cluster_pct"]
res = tukey_hsd(group0, group1, group2)
print(res)

In [ ]:
## Define the group you want to test first: 
testA = obs_df[obs_df["final_ann"] == "MG/BAMs"]
group0 = testA[testA["catpat"] == "NIC"]["cluster_pct"]
group1 = testA[testA["catpat"] == "IC"]["cluster_pct"]
group2 = testA[testA["catpat"] == "IC"]["cluster_pct"]
res = tukey_hsd(group0, group1, group2)
print(res)

In [ ]:
## Define the group you want to test first: 
testA = obs_df[obs_df["final_ann"] == "Monocytes"]
group0 = testA[testA["catpat"] == "NIC"]["cluster_pct"]
group1 = testA[testA["catpat"] == "IC"]["cluster_pct"]
group2 = testA[testA["catpat"] == "IC"]["cluster_pct"]
res = tukey_hsd(group0, group1, group2)
print(res)

In [ ]:
## Define the group you want to test first: 
testA = obs_df[obs_df["final_ann"] == "DCs"]
group0 = testA[testA["catpat"] == "NIC"]["cluster_pct"]
group1 = testA[testA["catpat"] == "IC"]["cluster_pct"]
group2 = testA[testA["catpat"] == "IC"]["cluster_pct"]
res = tukey_hsd(group0, group1, group2)
print(res)

In [ ]:
## Define the group you want to test first: 
testA = obs_df[obs_df["final_ann"] == "CD20+ B cells"]
group0 = testA[testA["catpat"] == "NIC"]["cluster_pct"]
group1 = testA[testA["catpat"] == "IC"]["cluster_pct"]
group2 = testA[testA["catpat"] == "IC"]["cluster_pct"]
res = tukey_hsd(group0, group1, group2)
print(res)

In [ ]:
## Define the group you want to test first: 
testA = obs_df[obs_df["final_ann"] == "Plasmablasts"]
group0 = testA[testA["catpat"] == "NIC"]["cluster_pct"]
group1 = testA[testA["catpat"] == "IC"]["cluster_pct"]
group2 = testA[testA["catpat"] == "IC"]["cluster_pct"]
res = tukey_hsd(group0, group1, group2)
print(res)

In [ ]:
## Define the group you want to test first: 
testA = obs_df[obs_df["final_ann"] == "NK cells"]
group0 = testA[testA["catpat"] == "NIC"]["cluster_pct"]
group1 = testA[testA["catpat"] == "IC"]["cluster_pct"]
group2 = testA[testA["catpat"] == "IC"]["cluster_pct"]
res = tukey_hsd(group0, group1, group2)
print(res)

In [ ]:
## Define the group you want to test first: 
testA = obs_df[obs_df["final_ann"] == "pDCs"]
group0 = testA[testA["catpat"] == "NIC"]["cluster_pct"]
group1 = testA[testA["catpat"] == "IC"]["cluster_pct"]
group2 = testA[testA["catpat"] == "IC"]["cluster_pct"]
res = tukey_hsd(group0, group1, group2)
print(res)

# Annotations

In [ ]:
# Proteins to transfer
proteins = [
    "CD45RA_adt",
    "CD4_adt",
    "CD8_adt",
    "CD62L_adt",
    "CD95_adt",
]

# Extract protein expression matrix
prot_df = pd.DataFrame(
    sdata["prot"][:, proteins].X.toarray()
    if hasattr(sdata["prot"].X, "toarray") else sdata["prot"][:, proteins].X,
    index=sdata["prot"].obs_names,
    columns=[f"prot_{p}" for p in proteins]
)

# Transfer into gex.obs
sdata["gex"].obs = sdata["gex"].obs.join(prot_df)

# Metadata graphs

In [ ]:
## Load data
df = pd.read_excel(
    r"..\..\03_annotation\10_metadata_patients\Metadates_TCR_TT_CMV_EBV_Titer.xlsx"
)

#Group annotation
patient_list = ["Pt.11", "Pt.12", "Pt.14", "Pt.17", "Pt.20"]
control_list = ["Pt.1", "Pt.8", "Pt.18", "Pt.19", "Pt.21"]
IC_list = ["Pt.5", "Pt.4"]

#Match the lists defined above to define catpat
df["Catpat"] = pd.Categorical(np.where(df["Patient ID"].isin(patient_list), "MS", np.where(df["Patient ID"].isin(control_list), "NIC", np.where(df["Patient ID"].isin(IC_list), "IC", "Error"))))

# Build the data frame
plot_df = df[["Patient ID", "Catpat", "Age", "Gender",
   ]].drop_duplicates()

plt.figure(figsize=(1, 2))

# Boxplot (controls box width)
sb.boxplot(
    data=plot_df,
    x="Catpat",
    y="Age",
    order=["NIC", "IC", "MS"],
    width=0.6,          # ⬅️ box width
    showfliers=False, 
    boxprops=dict(facecolor="none"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black")
)

# dots
sb.stripplot(
    data=plot_df,
    x="Catpat",
    y="Age",
    order=["NIC", "IC", "MS"],
    jitter=0.15,
    size=5,          # ⬅️ larger dots
    color="black",   # ⬅️ black points
    alpha=0.8
)

plt.ylabel("Age")
plt.xlabel("")
plt.savefig(
     "../../05_plots/13_final_plots/38_age_distribution.pdf", 
    dpi=300,
    format="pdf", 
    bbox_inches="tight"
)
plt.show()

In [ ]:
plt.figure(figsize=(1, 2))

sb.countplot(
    data=plot_df,
    x="Catpat",
    hue="Gender",
    order=["NIC", "IC", "MS"],
    hue_order=["female", "male"],
    legend=False
)

plt.ylabel("Number of patients")
plt.xlabel("")  # removes the x-axis title
plt.savefig(
     "../../05_plots/13_final_plots/38_gender_count.pdf", 
    dpi=300,
    format="pdf", 
    bbox_inches="tight"
)
plt.show()

In [ ]:
## Load data
df = pd.read_excel(
    r"..\..\03_annotation\10_metadata_patients\Metadates_TCR_TT_CMV_EBV_Titer.xlsx"
)


#Group annotation
patient_list = ["Pt.11", "Pt.12", "Pt.14", "Pt.17", "Pt.20"]
control_list = ["Pt.1", "Pt.8", "Pt.18", "Pt.19", "Pt.21"]
IC_list = ["Pt.5", "Pt.4"]
#Match the lists defined above to define catpat
df["Catpat"] = pd.Categorical(np.where(df["Patient ID"].isin(patient_list), "MS", np.where(df["Patient ID"].isin(control_list), "NIC", np.where(df["Patient ID"].isin(IC_list), "IC", "Error"))))

#Process
df["Ig_EBNA"] = df["EBV-EBNA-IgG Ergebnis"]
df["Ig_VCA"] = df["EBV-VCA-IgG Ergebnis"]
df["Im_VCA"] = df["EBV-VCA-IgM Ergebnis"]

#Remove the missing Pt.1
df = df[df["Ig_EBNA"] != "missing "]
df["Ig_CMV"] = df["CMV-IgG Ergebnis"]

# Copy column
df["Ig_CMV"] = df["CMV-IgG Ergebnis"]
# Replace "< 10" with random integers 0–9
mask = df["Ig_CMV"] == "<10,0"
df.loc[mask, "Ig_CMV"] = np.random.randint(0, 10, size=mask.sum())

# Build the data frame
plot_df = df[["Patient ID", "Catpat", "Ig_EBNA", "Ig_VCA", "Im_VCA", "Ig_CMV"
   ]].drop_duplicates()

# Process
plot_df["Ig_EBNA"] = plot_df["Ig_EBNA"].astype(float)
plot_df["Ig_VCA"] = plot_df["Ig_VCA"].astype(float)
plot_df["Im_VCA"] = plot_df["Im_VCA"].astype(float)
plot_df["Ig_CMV"] = plot_df["Ig_CMV"].astype(float)

In [ ]:
plt.figure(figsize=(1, 2))

# Boxplot (controls box width)
sb.boxplot(
    data=plot_df,
    x="Catpat",
    y="Ig_EBNA",
    order=["NIC", "IC", "MS"],
    width=0.6,          # ⬅️ box width
    showfliers=False, 
    boxprops=dict(facecolor="none"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black")
)

# dots
sb.stripplot(
    data=plot_df,
    x="Catpat",
    y="Ig_EBNA",
    order=["NIC", "IC", "MS"],
    jitter=0.15,
    size=5,          # ⬅️ larger dots
    color="black",   # ⬅️ black points
    alpha=0.8
)

plt.ylabel("IgG EBNA")
plt.xlabel("")
plt.savefig(
     "../../05_plots/13_final_plots/38_IgG_EBNA1.pdf", 
    dpi=300,
    format="pdf", 
    bbox_inches="tight"
)
plt.show()

In [ ]:
plt.figure(figsize=(1, 2))

# Boxplot (controls box width)
sb.boxplot(
    data=plot_df,
    x="Catpat",
    y="Ig_VCA",
    order=["NIC", "IC", "MS"],
    width=0.6,          # ⬅️ box width
    showfliers=False, 
    boxprops=dict(facecolor="none"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black")
)

# dots
sb.stripplot(
    data=plot_df,
    x="Catpat",
    y="Ig_VCA",
    order=["NIC", "IC", "MS"],
    jitter=0.15,
    size=5,          # ⬅️ larger dots
    color="black",   # ⬅️ black points
    alpha=0.8
)

plt.ylabel("IgG VCA")
plt.xlabel("")
plt.savefig(
      "../../05_plots/13_final_plots/38_IgG_VCA.pdf", 
    dpi=300,
    format="pdf", 
    bbox_inches="tight"
)
plt.show()

In [ ]:
plt.figure(figsize=(1, 2))

# Boxplot (controls box width)
sb.boxplot(
    data=plot_df,
    x="Catpat",
    y="Im_VCA",
    order=["NIC", "IC", "MS"],
    width=0.6,          # ⬅️ box width
    showfliers=False, 
    boxprops=dict(facecolor="none"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black")
)

# dots
sb.stripplot(
    data=plot_df,
    x="Catpat",
    y="Im_VCA",
    order=["NIC", "IC", "MS"],
    jitter=0.15,
    size=5,          # ⬅️ larger dots
    color="black",   # ⬅️ black points
    alpha=0.8
)

plt.ylabel("IgM VCA")
plt.xlabel("")
plt.savefig(
     "../../05_plots/13_final_plots/38_IgM_VCA.pdf", 
    dpi=300,
    format="pdf", 
    bbox_inches="tight"
)
plt.show()

In [ ]:
plt.figure(figsize=(1, 2))

# Boxplot (controls box width)
sb.boxplot(
    data=plot_df,
    x="Catpat",
    y="Ig_CMV",
    order=["NIC", "IC", "MS"],
    width=0.6,          # ⬅️ box width
    showfliers=False, 
    boxprops=dict(facecolor="none"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    medianprops=dict(color="black")
)

# dots
sb.stripplot(
    data=plot_df,
    x="Catpat",
    y="Ig_CMV",
    order=["NIC", "IC", "MS"],
    jitter=0.15,
    size=5,          # ⬅️ larger dots
    color="black",   # ⬅️ black points
    alpha=0.8
)

#Enforce detection thresholds
plt.ylim(-20, 600)
plt.ylabel("IgG CMV")
plt.xlabel("")
plt.savefig(
     "../../05_plots/13_final_plots/38_IgG_CMV.pdf", 
    dpi=300,
    format="pdf", 
    bbox_inches="tight"
)
plt.show()

# Supporting information

### Number of cells 

In [ ]:
sdata["gex"].obs.groupby("Compartment").size()

## MAIT expression

In [ ]:
MAIT_dat = sdata.copy()
mu.pp.filter_obs(MAIT_dat["gex"], "final_ann", lambda x: x.isin(["CD4 T cells", "CD8 T cells", "MAIT cells"]))
mu.pp.filter_obs(MAIT_dat["gex"], "Compartment", lambda x: x == "PB")
MAIT_dat.update()


In [ ]:
# Define desired order
order = ["CD4 T cells", "CD8 T cells", "MAIT cells"]

# Set categorical ordering
MAIT_dat["gex"].obs["final_ann"] = MAIT_dat["gex"].obs["final_ann"].cat.remove_unused_categories()
MAIT_dat["gex"].obs["final_ann"] = pd.Categorical(
    MAIT_dat["gex"].obs["final_ann"],
    categories=order,
    ordered=True
)


# comp_palette = {"CSF": "#4B8AC1", "PB": "#C06A6A"}

fig = sc.pl.violin(
    MAIT_dat["gex"],
    keys=["TRAV1-2", "SLC4A10", "KLRB1"],
    size=0,
    groupby="final_ann",
    hue="final_ann",
    linewidth = 0.5,
    # palette=comp_palette,
    rotation=90,
    show=False,
)

# Rasterize only the dot collections
for ax in plt.gcf().axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

# set figure size HERE
plt.gcf().set_size_inches(4, 1.5)   # width, height in inches

#Save the figure
for ax in plt.gcf().axes:
    ax.grid(False)

    legend = ax.get_legend()
    if legend is not None:
        legend.remove()

sb.despine()
plt.tight_layout()

# correct way to save
plt.gcf().savefig(
    "../../05_plots/13_final_plots/40_violin_MAIT.pdf",
    dpi=400,
    format="pdf",
)

plt.show()

## Cell counts per donor per ml CSF after processing

In [ ]:
## Import volumnes per donor
Puncture_amount = {
    "Pt1": 20,
    "Pt4": 11,
    "Pt5": 8,
    "Pt8": 10,
    "Pt11": 15,
    "Pt12": 15,
    "Pt14": 14,
    "Pt17": 14,
    "Pt18": 12,
    "Pt19": 14,
    "Pt20": 14.5,
    "Pt21": 14
}

In [ ]:
obs_df = sdata["gex"].obs.copy()

## count cells per donor per compartment
plot_df = obs_df[["donor", "catpat", "Compartment"]].value_counts().reset_index()

## Puncture amount
plot_df["csf_volume"] = plot_df["donor"].map(Puncture_amount)

## normalized cell counts
plot_df["count_per_ml"] = plot_df["count"] / plot_df["csf_volume"]


In [ ]:
# Boxplot (controls box width)
g = sb.catplot(
    data=plot_df,
    x="catpat",
    y="count_per_ml",
    col="Compartment",
    order=["NIC", "IC", "MS"],
    kind="box",
    showfliers=False,
    width=0.6,
    height=2,
    aspect=0.7,
    sharey=True,
    boxprops=dict(facecolor="none", edgecolor="black"),
)
# Add dots
g.map_dataframe(
    sb.stripplot,
    x="catpat",
    y="count_per_ml",
    order=["NIC", "IC", "MS"],
    jitter=0.15,
    size=5,
    color="black",
    alpha=0.8,
)

# Formatting
for ax in g.axes.flat:
    ax.grid(False)
    ax.set_xlabel("")
    ax.set_title(ax.get_title().replace("Compartment = ", ""))
    ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=4, integer=True))

    legend = ax.get_legend()
    if legend is not None:
        legend.remove()

# set figure size HERE
plt.gcf().set_size_inches(4, 4)   # width, height in inches

sb.despine()

g.savefig(
    "../../05_plots/13_final_plots/40_TotalCellCount_per_compartment.pdf",
    dpi=400,
)

plt.show()

# Export PB/CSF cell counts

In [ ]:
obs_df = sdata["gex"].obs.copy()

## count cells per donor per compartment
plot_df = obs_df[["donor", "catpat", "Compartment", "final_ann"]].value_counts().reset_index()
export_df = plot_df.pivot_table(index = ["donor", "catpat", "Compartment"], columns="final_ann", values="count", fill_value=0, observed=False).reset_index().sort_values(["Compartment", "donor"])
export_df.to_excel("../../06_csv_outputs/Counts/Table_cell_counts.xlsx")

# CSF count for writing

In [ ]:
obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[obs_df["Compartment"]== "CSF"]

## count cells per donor per compartment
plot_df = obs_df[["donor", "catpat"]].value_counts().reset_index()
plot_df
# obs_df[obs_df["Compartment"] == "CSF"][["donor", "catpat"]].value_counts().reset_index()

In [ ]:
obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[obs_df["Compartment"]== "CSF"]

## count cells per donor per compartment
plot_df = obs_df[["donor", "catpat", "final_ann"]].value_counts().reset_index()
plot_df
# obs_df[obs_df["Compartment"] == "CSF"][["donor", "catpat"]].value_counts().reset_index()

# PB count for writing

In [ ]:
obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[obs_df["Compartment"]== "PB"]

## count cells per donor per compartment
plot_df = obs_df[["donor", "catpat", "final_ann"]].value_counts().reset_index()
plot_df.sort_values("donor")

## umaps

In [ ]:
# Define colors for Compartment
comp_palette = {"CSF": "#4B8AC1", "PB": "#C06A6A"}

# --- UMAP 1: CSF compartment ---
fig, ax = plt.subplots(figsize=(3.5, 3))
sc.pl.umap(
    sdata["gex"],
    color=["Compartment"],
    groups=["CSF"],
    size=10,
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    show=False,
    ax=ax,
    palette=comp_palette
)
# Rasterize points
for coll in ax.collections:
    coll.set_rasterized(True)

plt.savefig("../../05_plots/13_final_plots/39_CSF_umap.pdf",
            dpi=300, format="pdf", bbox_inches="tight")
plt.close()

# --- UMAP 2: PB compartment ---
fig, ax = plt.subplots(figsize=(3.5, 3))
sc.pl.umap(
    sdata["gex"],
    color=["Compartment"],
    groups=["PB"],
    size=10,
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    show=False,
    ax=ax,
    palette=comp_palette
)
for coll in ax.collections:
    coll.set_rasterized(True)

plt.savefig("../../05_plots/13_final_plots/39_PB_umap.pdf",
            dpi=300, format="pdf", bbox_inches="tight")
plt.close()

# --- UMAP 3: MAIT cells ---
fig, ax = plt.subplots(figsize=(3.5, 3))
sc.pl.umap(
    sdata["gex"],
    color=["final_ann"],
    groups=["MAIT cells"],
    size=10,
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    show=False,
    ax=ax
)
for coll in ax.collections:
    coll.set_rasterized(True)

plt.savefig("../../05_plots/13_final_plots/39_MAIT_umap.pdf",
            dpi=300, format="pdf", bbox_inches="tight")
plt.close()

# --- UMAP 4: chains ---
fig, ax = plt.subplots(figsize=(3.5, 3))
sc.pl.umap(
    sdata["gex"],
    color=["chain_pairing"],
    size=10,
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    show=False,
    ax=ax
)
for coll in ax.collections:
    coll.set_rasterized(True)

plt.savefig("../../05_plots/13_final_plots/39_TCR_umap.pdf",
            dpi=300, format="pdf", bbox_inches="tight")
plt.show()

# VDJ quantifications

In [ ]:
# --- Extract relevant data ---
df = sdata["gex"].obs[["Compartment", "chain_pairing", "final_ann"]].copy()

# Add new category "No VDJ" and fill NaNs
df.loc[:, "chain_pairing"] = df["chain_pairing"].cat.add_categories("No VDJ")
df.loc[:, "chain_pairing"] = df["chain_pairing"].fillna("No VDJ")

# Extract colors for chain_pairing
chain_colors = sdata["gex"].uns["chain_pairing_colors"]
categories = df["chain_pairing"].cat.categories

# Build a palette dictionary
chain_palette = dict(zip(categories[:len(chain_colors)], chain_colors))
chain_palette["No VDJ"] = "lightgrey"  # Add grey for missing

# --- Filter to CSF and T-cell clusters ---
df = df[df["Compartment"] == "CSF"].copy()

# --- Compute normalized proportions ---
counts = pd.crosstab(df["final_ann"], df["chain_pairing"])
props = counts.div(counts.sum(axis=1), axis=0)

# --- Plot manually as stacked bars ---
plt.figure(figsize=(3, 1.5))
bottom = np.zeros(len(props))

for col in props.columns:
    plt.bar(
        x=props.index,
        height=props[col],
        bottom=bottom,
        color=chain_palette[col],  # Use palette including "No VDJ"
        edgecolor="black",
        label=col
    )
    bottom += props[col].values

plt.ylabel("Proportion")
plt.xlabel("")
plt.xticks(rotation=90, ha="right")
plt.legend(title="Chain pairing", bbox_to_anchor=(1.05, 1), loc="upper left")
# plt.tight_layout()
plt.savefig(
    "../../05_plots/13_final_plots/41_vdj_boxplots.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()


## Gene count quantifications

In [ ]:
# Extract obs and sort groups by median
obs = sdata["gex"].obs
medians = obs.groupby("final_ann")["n_genes_by_counts"].median()
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (5, 2)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="n_genes_by_counts",
        groupby="final_ann",
        order=sorted_groups,
        palette=celltype_palette,
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,   # no individual dots
        jitter=False,      # optional, just to be clean
        inner="box"        # ⬅️ shows the small boxplot inside the violin
    )

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')

# Add horizontal reference line if needed
plt.axhline(y=200, linewidth=1.5, color="black", linestyle="--")
plt.ylabel("Genes per cell")

# Rasterize violin and box shapes for PDF
for col in ax.collections:
    col.set_rasterized(True)

plt.tight_layout()
plt.savefig(
    "../../05_plots/13_final_plots/42_GenesPerCell_ quantification_per_L2_box.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()

In [ ]:
# Compute medians per group
medians = sdata["gex"].obs.groupby("final_ann")["total_counts"].median()

# Sort groups by median (ascending)
sorted_groups = medians.sort_values().index.tolist()

# --- Plot violin with small boxplot inside ---
with plt.rc_context({"figure.figsize": (5, 2)}):
    ax = sc.pl.violin(
        sdata["gex"],
        keys="total_counts",
        groupby="final_ann",
        order=sorted_groups,  # sorted by median
        palette=celltype_palette,
        multi_panel=False,
        rotation=45,
        show=False,
        stripplot=False,      # no individual dots
        inner="box"           # small boxplot inside violin
    )

# Rotate x-axis labels
plt.xticks(rotation=45, ha='right')
plt.ylabel("Total read counts")

# Rasterize violin and box shapes for PDF
for col in ax.collections:
    col.set_rasterized(True)

# Save figure
plt.tight_layout()
plt.savefig(
   "../../05_plots/13_final_plots/43_TotalReads_quantification_per_ann_violin.pdf",
    bbox_inches="tight",
    dpi=300
)
plt.show()


# Doublet example (observed vs expected proportions)

In [ ]:
## Manually note down all cell numbers observed before pre-processing CellRanger outputs
raw_cells = pd.DataFrame({
    "batch": ["NK_1", "NK_3", "NK_5", "NK_6", "NK_7", "NK_9a", "NK_9b", "NK_10", "NK_11b"], 
    "Cells": [4730,    3324,   18365,   14887,   3631,    4966,    6067,   2208,    9295]
  
})

#Calculate the expected doublet rate (a fraction - not a percentage here!). Use the 10x genomics approximation of x0.0008% doublets per cell. 
raw_cells["expected_multiplet_rate"] = raw_cells["Cells"]*0.000008
raw_cells["expected_doublet_count"] = raw_cells["expected_multiplet_rate"]*raw_cells["Cells"]

## obs_df
obs_df = sdata["gex"].obs.copy()
obs_df = obs_df[obs_df["Compartment"] == "CSF"]
B_df = obs_df[obs_df["final_ann"].isin(["CD20+ B cells", "Plasmablasts"])][["donor", "catpat", "batch", "chain_pairing"]]
B_df["TCR_positive"] = np.where(B_df["chain_pairing"].isna(), "TCR-", "TCR+")

## Count cells
B_df = B_df.groupby(["batch", "donor", "catpat", "TCR_positive"], observed=True).size().reset_index(name="n_cells")

## Calculate frequencies within each donor × catpat
B_df["total_B_cells"] = B_df.groupby(["batch", "donor", "catpat"], observed=True)["n_cells"].transform("sum")
B_df["observed_proportion"] = B_df["n_cells"]/B_df["total_B_cells"]

## Get a df with the TCR+ frequencies per 
observed_df = B_df[B_df["TCR_positive"] == "TCR+"]

## Left join
merged_df = observed_df.merge(raw_cells, how="left")
merged_df = merged_df[~merged_df["expected_doublet_count"].isna()]

## Make longer
merged_long = merged_df.melt(
    id_vars=["donor", "catpat"],
    value_vars=["observed_proportion", "expected_multiplet_rate"],
    var_name="metric",
    value_name="value"
)

## Plot "metric" as x-axis and "value" as y-axis. Connect dots from the same donor x catpat combination
plt.figure(figsize=(2.5, 3))

# unique pair identifier
merged_long["pair"] = (
    merged_long["donor"].astype(str)
    + "_"
    + merged_long["catpat"].astype(str)
)

# connect points
sb.lineplot(
    data=merged_long,
    x="metric",
    y="value",
    units="pair",
    estimator=None,
    color="lightgrey",
    linewidth=1,
    alpha=0.8,
)

# draw points
sb.stripplot(
    data=merged_long,
    x="metric",
    y="value",
    hue="catpat",  # optional
    dodge=False,
    size=5,
    zorder=10,
)

plt.ylabel("Proportion")
plt.xlabel("")

sb.despine()
plt.tight_layout()

plt.savefig(
    "../../05_plots/13_final_plots/45_multiplet_rate_comparison.pdf",
    dpi=400,
)

plt.show()

# Doublet score

In [ ]:
sdata["gex"].obs.head(2)

In [ ]:
# --- Set scrublet_score to NaN for PB cells ---
sdata["gex"].obs.loc[sdata["gex"].obs["Compartment"] == "PB", "scrublet_score"] = np.nan


# --- UMAP Scrublet score ---
fig, ax = plt.subplots(figsize=(3.5, 3))
sc.pl.umap(
    sdata["gex"],
    color=["scrublet_score"],
    size=10,
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    show=False,
    ax=ax,
    color_map='RdYlBu_r'
)

# Rasterize points
for coll in ax.collections:
    coll.set_rasterized(True)

# Save figure
plt.savefig(
    "../../05_plots/13_final_plots/44_ScrubletScores_CSF_without_recalculating_globally.pdf",
    dpi=300,
    format="pdf",
    bbox_inches="tight"
)
plt.show()


In [ ]:
#Recalculate scrublet on a global level.
csf_dat = sdata.copy()
mu.pp.filter_obs(csf_dat["gex"], "Compartment", lambda x: (x == "CSF")) 

## We need clarify for how many cells we detected in each case
N_cells = csf_dat["gex"].obs.groupby("donor").size()
print("Mean cell number was: " + str(N_cells.mean()) + " cells")

In [ ]:
### So we anticipate that 6241.5 cells per sample should give a doublet rate of 0.049932
N_cells.mean() * 0.0008 / 100

In [ ]:
scrub = scr.Scrublet(csf_dat["gex"].layers["counts"], expected_doublet_rate=0.049932)

In [ ]:
doublet_scores, predicted_doublets = scrub.scrub_doublets(min_counts=2, 
                                                          min_cells=3, 
                                                          min_gene_variability_pctl=85, 
                                                          n_prin_comps=30)

In [ ]:
scrub.plot_histogram();

In [ ]:
## Get annotation list
# Doublet scores per cell
scores = scrub.doublet_scores_obs_

# Boolean array of doublet calls (indexed to your input matrix)
predicted = scrub.predicted_doublets_

# Check first 10
print(predicted[:10])

In [ ]:
## Use the bolean list to annotate csf cells
csf_dat["gex"].obs["scrublet_score"] = scores 
csf_dat["gex"].obs["scrublet_pred"] = np.where(predicted == False, "Singlet", "Multiplet")

In [ ]:
# Create a new column in sdata["gex"].obs filled with NaN
sdata["gex"].obs["scrublet_score"] = np.nan

# Transfer values from csf_dat to the matching indexes
common_idx = sdata["gex"].obs.index.intersection(csf_dat["gex"].obs.index)
sdata["gex"].obs.loc[common_idx, "scrublet_score"] = csf_dat["gex"].obs.loc[common_idx, "scrublet_score"]

In [ ]:
# --- Set scrublet_score to NaN for PB cells ---
sdata["gex"].obs.loc[sdata["gex"].obs["Compartment"] == "PB", "scrublet_score"] = np.nan

# Optional: define compartment colors if needed
comp_palette = {"CSF": "skyblue", "PB": "salmon"}

# --- UMAP Scrublet score ---
fig, ax = plt.subplots(figsize=(3.5, 3))
sc.pl.umap(
    sdata["gex"],
    color=["scrublet_score"],
    size=10,
    legend_fontweight='bold',
    frameon=False,
    vmin=0,
    legend_fontoutline=2,
    show=False,
    ax=ax,
    color_map='RdYlBu_r'
)

# Rasterize points
for coll in ax.collections:
    coll.set_rasterized(True)

# Save figure
plt.savefig(
    "../../05_plots/13_final_plots/44_ScrubletScores_CSF_without_recalculated_globally.pdf",
    dpi=300,
    format="pdf",
    bbox_inches="tight"
)
plt.show()


# 